In [ ]:
# Multi-Period Capital Investment (0-1) — PuLP
import pulp as pl
# investment: (period, cost, payout-next-period)
inv = {"A":(1,10000,15000),"B":(1,20000,30000),"C":(1,15000,22000),
       "D":(2,12000,18000),"E":(2,25000,35000),"F":(2,18000,26000),
       "G":(3,15000,20000),"H":(3,10000,14000),"I":(3,20000,28000),
       "J":(4, 8000,10000),"K":(4,16000,22000),"L":(4,12000,16000)}
START = 50000
prob = pl.LpProblem("Multi_Period_Investment", pl.LpMaximize)
x = {i: pl.LpVariable(f"x_{i}", cat="Binary") for i in inv}
# objective: cash from period-4 payouts
prob += pl.lpSum(p*x[i] for i,(t,c,p) in inv.items() if t==4), "Final_cash"
for period in (1,2,3,4):
    cost_t = pl.lpSum(c*x[i] for i,(t,c,p) in inv.items() if t==period)
    if period==1:
        prob += cost_t <= START, "Budget_1"
    else:
        prev_payout = pl.lpSum(p*x[i] for i,(t,c,p) in inv.items() if t==period-1)
        prob += cost_t <= prev_payout, f"Budget_{period}"
prob.solve()
print("Status:", pl.LpStatus[prob.status])
print("Chosen:", [i for i in inv if x[i].value()==1])
print("Final cash =", pl.value(prob.objective))
